In [1]:
# from CSVtoSQL import CSVFolderToSQL
# CSVFolderToSQL("../synthea/output/csv", "medical_data.db").csv_to_sql()

In [2]:
import pandas as pd
import numpy as np
from SQL_query import SQLQuery
SQL = SQLQuery("medical_data.db")
query = """
WITH LATEST_PREGNANCY AS (
    SELECT
        PATIENT,
        MAX(DATE(START)) AS PREGNANCY_START,
        DATE(MAX(START), '+270 days') AS EST_DELIVERY_DATE
    FROM encounters
    WHERE DESCRIPTION = 'Prenatal initial visit (regime/therapy)'
    GROUP BY PATIENT
),
LATEST_PREG_DIA AS (
    SELECT
        o.PATIENT,
        MAX(DATE(o.DATE)) AS OBSERVATION_DATE,
        MAX(CASE WHEN o.DESCRIPTION = 'Diastolic Blood Pressure' THEN o.VALUE END) AS Diastolic_Blood_Pressure
    FROM observations as o
    INNER JOIN LATEST_PREGNANCY as lp ON o.PATIENT = lp.PATIENT
    WHERE o.DESCRIPTION = 'Diastolic Blood Pressure' AND DATE(o.DATE) <= lp.EST_DELIVERY_DATE 
    GROUP BY o.PATIENT
),
LATEST_PREG_SYS AS (
    SELECT
        o.PATIENT,
        MAX(DATE(o.DATE)) AS OBSERVATION_DATE,
        MAX(CASE WHEN o.DESCRIPTION = 'Systolic Blood Pressure' THEN o.VALUE END) AS Systolic_Blood_Pressure
    FROM observations as o
    INNER JOIN LATEST_PREGNANCY as lp ON o.PATIENT = lp.PATIENT
    WHERE o.DESCRIPTION = 'Systolic Blood Pressure' AND DATE(o.DATE) <= lp.EST_DELIVERY_DATE
    GROUP BY o.PATIENT
),
DEATH_CAUSE AS (
    SELECT
        d.PATIENT,
        d.VALUE AS CAUSE_OF_DEATH
    FROM observations as d
    WHERE d.DESCRIPTION = 'Cause of Death [US Standard Certificate of Death]'
)
SELECT 
    t1.Id,
    DATE(t1.BIRTHDATE) AS BIRTHDATE,
    DATE(t1.DEATHDATE) AS DEATHDATE,
    t1.RACE,
    t1.ETHNICITY,
    t1.GENDER,
    t1.COUNTY,
    t1.HEALTHCARE_EXPENSES,
    t1.HEALTHCARE_COVERAGE,
    t1.INCOME,
    t1.HEALTHCARE_EXPENSES - t1.INCOME AS RELATIVE_EXPENSES,
    DATE(t2.PREGNANCY_START) AS PREGNANCY_START,
    DATE(t2.EST_DELIVERY_DATE) AS EST_DELIVERY_DATE,
    DATE(t3.OBSERVATION_DATE) AS BP_OBSERVATION_DATE,
    t3.Diastolic_Blood_Pressure,
    t4.Systolic_Blood_Pressure,
    t8.CAUSE_OF_DEATH
    
FROM patients as t1
INNER JOIN LATEST_PREGNANCY as t2 ON t1.Id = t2.PATIENT
LEFT JOIN LATEST_PREG_DIA as t3 ON t1.Id = t3.PATIENT
LEFT JOIN LATEST_PREG_SYS as t4 ON t1.Id = t4.PATIENT
LEFT JOIN DEATH_CAUSE as t8 ON t1.Id = t8.PATIENT
"""
LAfemMed = SQL.execute_query(query)
LAfemMed['BIRTHDATE'] = pd.to_datetime(LAfemMed['BIRTHDATE'])
LAfemMed['DEATHDATE'] = pd.to_datetime(LAfemMed['DEATHDATE'])
LAfemMed['PREGNANCY_START'] = pd.to_datetime(LAfemMed['PREGNANCY_START'])
LAfemMed['EST_DELIVERY_DATE'] = pd.to_datetime(LAfemMed['EST_DELIVERY_DATE'])
LAfemMed['BP_OBSERVATION_DATE'] = pd.to_datetime(LAfemMed['BP_OBSERVATION_DATE'])
LAfemMed['Days_Pregancy_to_Death'] = (LAfemMed['DEATHDATE'] - LAfemMed['PREGNANCY_START']).dt.days
LAfemMed['Maternal_Mortality'] = np.where(LAfemMed['Days_Pregancy_to_Death'] <= 270+42, 1, 0)

Maternal_Deaths = LAfemMed[LAfemMed['Maternal_Mortality'] == 1]

# Preprocess Data

In [3]:
from sklearn.preprocessing import StandardScaler
numeric_cols = ['INCOME', 'RELATIVE_EXPENSES', 'Diastolic_Blood_Pressure', 'Systolic_Blood_Pressure']
all_cols = ['INCOME', 'RELATIVE_EXPENSES', 'Diastolic_Blood_Pressure', 'Systolic_Blood_Pressure', 'RACE', 'Maternal_Mortality']
final_data = LAfemMed[all_cols].dropna()
race_encoded = pd.get_dummies(final_data['RACE'])
scale = StandardScaler()
numeric_scaled = scale.fit_transform(final_data[numeric_cols])
X = np.column_stack([race_encoded.values, numeric_scaled]).astype(float)
y = final_data['Maternal_Mortality'].to_numpy().astype(float)

In [4]:
from NNmod import NN_mod
nn_mort = NN_mod(epochs=1000)
nn_mort.fit(X, y)

Epoch: 4, Loss: 0.6240
Epoch: 9, Loss: 0.5199
Epoch: 14, Loss: 0.4197
Epoch: 19, Loss: 0.3303
Epoch: 24, Loss: 0.2562
Epoch: 29, Loss: 0.1992
Epoch: 34, Loss: 0.1583
Epoch: 39, Loss: 0.1294
Epoch: 44, Loss: 0.1095
Epoch: 49, Loss: 0.0958
Epoch: 54, Loss: 0.0860
Epoch: 59, Loss: 0.0794
Epoch: 64, Loss: 0.0749
Epoch: 69, Loss: 0.0716
Epoch: 74, Loss: 0.0691
Epoch: 79, Loss: 0.0670
Epoch: 84, Loss: 0.0653
Epoch: 89, Loss: 0.0641
Epoch: 94, Loss: 0.0630
Epoch: 99, Loss: 0.0619
Epoch: 104, Loss: 0.0609
Epoch: 109, Loss: 0.0602
Epoch: 114, Loss: 0.0594
Epoch: 119, Loss: 0.0587
Epoch: 124, Loss: 0.0581
Epoch: 129, Loss: 0.0575
Epoch: 134, Loss: 0.0570
Epoch: 139, Loss: 0.0565
Epoch: 144, Loss: 0.0560
Epoch: 149, Loss: 0.0556
Epoch: 154, Loss: 0.0552
Epoch: 159, Loss: 0.0549
Epoch: 164, Loss: 0.0545
Epoch: 169, Loss: 0.0542
Epoch: 174, Loss: 0.0539
Epoch: 179, Loss: 0.0536
Epoch: 184, Loss: 0.0533
Epoch: 189, Loss: 0.0530
Epoch: 194, Loss: 0.0527
Epoch: 199, Loss: 0.0525
Epoch: 204, Loss: 0.05